## Example submission for the CausalRivers leaderboard

Please follow the instructions below to generate a submission that holds your calculated performance
We might update this process in the future to allow for the submission of raw predictions, but for now, you can submit your performance scores directly.
Note that you don't have to submit solutions for each available dataset.

In [5]:
import json
from pathlib import Path

import pandas as pd

report_relpath = Path("benchmarks/causal_discovery/benchmarking-results.json")
cwd = Path.cwd()
project_root = next(
    (
        directory
        for directory in (cwd, *cwd.parents)
        if (directory / report_relpath).is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start Jupyter from within the GP-Mining checkout.")
benchmark_dir = project_root / report_relpath.parent


### Generate new results

From the GP-Mining repository root, run the benchmark once for each label file you want to score.

~~~bash
python benchmarks/causal_discovery/causalrivers/benchmark.py label_path=datasets/confounder_3/east.p
~~~

The next cell builds the submission from the archived scoring files listed in the benchmark export. Use the new scoring paths to submit a new run.


In [6]:
report_path = benchmark_dir / "benchmarking-results.json"
entries = json.loads(report_path.read_text(encoding="utf-8"))["causalrivers"]
scores = []
for entry in entries:
    scoring = pd.read_csv(benchmark_dir / entry["notebook_scoring_path"], index_col=0)
    scores.append(scoring.loc["Individual AUROC"].iloc[0])

df_scorings = pd.DataFrame(
    {"TGRAANK": scores}, index=[entry["graph_set"] for entry in entries]
)
df_scorings.T.to_csv(benchmark_dir / "causalrivers/tgraank_submission.csv")
